<div style='background:#000000;border-radius:10px;padding:26px 30px;border-top:6px solid #3777FF;border-bottom:3px solid #0EE071;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'>
<p style='color:#0EE071;margin:0;letter-spacing:2px;font-size:12px'>NOVA IMS · ENTERPRISE DATA SCIENCE BOOTCAMP 2026 · PROJETO D</p>
<h1 style='color:#FFFFFF;margin:6px 0 4px 0'>Telco Customer Churn — <span style='color:#3777FF'>Retention Management</span></h1>
<p style='color:#FFFFFF;margin:0;font-size:15px'>Notebook 01 · União dos dados, análise exploratória e pré-processamento para ensemble</p>
</div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>Índice · Estrutura do notebook</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Objetivo: preparar uma base <b style='color:#0EE071'>limpa, validada e documentada</b> para treinar modelos ensemble que estimem o risco de churn e
apoiem o Head of Customer Retention a decidir <b style='color:#3777FF'>quem deve receber uma intervenção de retenção</b>.</p>
<ol style='margin:4px 0'>
<li>Setup do ambiente</li><li>Carregamento dos ficheiros</li><li>Validação de chaves</li>
<li>União dos datasets</li><li>Qualidade de dados e derivações</li><li>População e target</li>
<li>Papéis das variáveis</li><li>Variáveis numéricas</li><li>Variáveis categóricas</li>
<li>Associação com o target e leakage</li><li>Feature engineering</li>
<li>Split e pré-processadores</li><li>Exportação</li><li>Próximos passos</li></ol></div></div>

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>1 · Setup do ambiente</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Localiza a raiz do repositório, adiciona <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>src/</code> ao caminho e importa os módulos do projeto.
O tema visual (fundo <b style='color:#3777FF'>preto</b>, destaque <b style='color:#3777FF'>azul #3777FF</b>, acento <b style='color:#0EE071'>verde #0EE071</b>) é aplicado a todos os gráficos.</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>config</code> — caminhos e parâmetros (seed, % de teste)</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data</code> — carregar, validar e unir</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>features</code> — população, papéis das variáveis, feature engineering</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>analysis</code> / <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>plots</code> — estatísticas e gráficos</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing</code> — split, transformadores e exportação</li></ul></div></div>

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import warnings
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

from telco_churn import analysis, data, features, plots, preprocessing
from telco_churn.config import RANDOM_STATE, TAB_DIR, TARGET, ensure_dirs
from telco_churn.style import apply_theme, style_table

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
SHOW = True          # False para correr sem mostrar gráficos (são sempre guardados)

ensure_dirs()
apply_theme()
print("Raiz do projeto:", ROOT)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>2 · Carregamento dos ficheiros</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Lê os 5 ficheiros Excel de <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/raw/</code>. O parâmetro <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>keep_default_na=False</code> é <b style='color:#0EE071'>obrigatório</b>:
em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Offer</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Internet Type</code> o texto <i>None</i> é um valor real (sem oferta / sem internet) e não um valor em falta.</p></div></div>

In [ ]:
raw = data.load_raw()
display(style_table(pd.DataFrame({n: d.shape for n, d in raw.items()},
                                 index=["linhas", "colunas"]).T))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>3 · Validação de chaves e granularidade</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Antes de unir, confirmamos (em vez de assumir) que:</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Customer ID</code> é único em Demographics, Location, Services e Status</li><li>os 4 ficheiros contêm exatamente o mesmo conjunto de clientes</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Zip Code</code> é único em Population e todos os ZIP dos clientes têm correspondência</li></ul>
<p>Se alguma condição falhar, o código pára com um <b style='color:#0EE071'>assert</b>.</p></div></div>

In [ ]:
keys = data.validate_keys(raw)
display(style_table(keys))
print(f"Mesmos Customer ID nos 4 ficheiros: {keys.attrs['same_ids']}")
print(f"Cobertura Zip Code location -> population: {keys.attrs['zip_coverage']:.1%}")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>4 · União dos datasets</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Juntamos as 4 tabelas por <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Customer ID</code> (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>validate="one_to_one"</code>) e acrescentamos a população da área por
<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Zip Code</code> (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>validate="many_to_one"</code>). As colunas <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Count</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Quarter</code> são removidas antes, porque se repetem e não trazem informação.</p>
<p>Resultado esperado: <b style='color:#0EE071'>7.043 linhas, uma por cliente</b>.</p></div></div>

In [ ]:
df = data.merge_datasets(raw)
print(f"Dataset unificado: {df.shape[0]:,} clientes × {df.shape[1]} colunas")
df.head()

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>5 · Qualidade de dados e derivações</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Para cada coluna: tipo, nº de valores distintos, missing e strings vazias.
As strings vazias em <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Churn Category</code> / <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Churn Reason</code> são esperadas: só existem para quem saiu.</p>
<p>Depois verificamos as <b style='color:#3777FF'>derivações exatas</b> documentadas no guia — servem para justificar a remoção de colunas redundantes.</p></div></div>

In [ ]:
quality = data.data_quality(df)
quality.to_csv(TAB_DIR / "01_qualidade_dados.csv")
display(style_table(quality))

In [ ]:
checks = data.derivation_checks(df).to_frame()
display(style_table(checks, highlight="pct_concordancia"))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>6 · População de modelação e target</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>De acordo com o guia, o modelo core compara apenas <b style='color:#3777FF'>Churned</b> e <b style='color:#3777FF'>Stayed</b>.
Os clientes <b style='color:#0EE071'>Joined</b> ficam guardados à parte para scoring posterior.</p>
<p>Target: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>churn</code> = 1 se Churned, 0 se Stayed.</p></div></div>

In [ ]:
display(style_table(df["Customer Status"].value_counts().to_frame("n")))

df_model, df_joined = features.split_population(df)
overall_rate = df_model[TARGET].mean()
print(f"População de modelação: {len(df_model):,} clientes | taxa de churn = {overall_rate:.1%}")
plots.plot_target(df_model[TARGET], show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>7 · Papéis das variáveis</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Cada coluna recebe um papel explícito — esta tabela é exportada para documentar as decisões no relatório:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#0EE071'>leakage</b> — codificam o resultado ou só existem depois do churn: nunca entram no modelo</li><li><b style='color:#0EE071'>derivada de modelo</b> — <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Churn Score</code> (benchmark a bater) e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>CLTV</code> (priorização por valor)</li><li><b style='color:#0EE071'>suspeita de leakage</b> — <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> (ver secção 10)</li><li><b style='color:#0EE071'>redundante</b> — constantes, duplicadas, derivações exatas ou alta cardinalidade (<code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code>)</li><li><b style='color:#3777FF'>numérica</b> / <b style='color:#3777FF'>categórica</b> — candidatas a features</li></ul></div></div>

In [ ]:
NUM_COLS, CAT_COLS, roles = features.feature_roles(df_model)
roles.to_csv(TAB_DIR / "02_papeis_variaveis.csv", index=False)

summary = roles.groupby("papel")["variavel"].agg(n="size", variaveis=", ".join)
display(style_table(summary))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>8 · Variáveis numéricas</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Três perspetivas: perfil estatístico, distribuição por target e correlação entre variáveis.</p></div></div>

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>8.1 · Perfil estatístico</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Além do <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>describe()</code>, o perfil inclui:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#0EE071'>skew</b> — assimetria; acima de ±1 indica cauda longa</li><li><b style='color:#0EE071'>pct_zeros</b> — % de clientes com valor exatamente 0. Valores altos indicam variáveis <i>inflacionadas de zeros</i>, onde o zero significa ausência (ex.: nunca teve reembolso); uma flag binária pode ser mais informativa do que o montante</li><li><b style='color:#0EE071'>pct_outliers_IQR</b> — % fora de [Q1 − 1,5×IQR ; Q3 + 1,5×IQR] (regra de Tukey). Um outlier estatístico não é um erro: o pipeline mede, não remove. Nas árvores o impacto é mínimo</li></ul>
<p>Atenção: com muitos zeros, Q1 = Q3 = 0 e qualquer valor positivo surge como outlier — ler as duas métricas em conjunto.</p></div></div>

In [ ]:
num_profile = analysis.numeric_profile(df_model, NUM_COLS)
num_profile.to_csv(TAB_DIR / "03_perfil_numericas.csv")
display(style_table(num_profile, precision=2, highlight="pct_zeros"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>8.2 · Distribuições por target</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Histograma de densidade (<b style='color:#3777FF'>azul = Stayed</b>, <b style='color:#0EE071'>verde = Churned</b>) e boxplot. Curvas bem separadas indicam variáveis com poder discriminante;
curvas sobrepostas indicam pouca informação isolada.</p></div></div>

In [ ]:
for col in NUM_COLS:
    plots.plot_numeric(df_model, col, TARGET, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>8.3 · Correlação entre numéricas</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Spearman (baseada em ordens, robusta a assimetrias e outliers). Escala <b style='color:#3777FF'>azul</b> = negativa, <b style='color:#0EE071'>verde</b> = positiva.
Pares com |ρ| ≥ 0,8 indicam multicolinearidade — irrelevante para ensembles de árvores, relevante para modelos lineares.</p></div></div>

In [ ]:
corr = df_model[NUM_COLS + [TARGET]].corr(method="spearman")
plots.plot_correlation(corr, show=SHOW)
display(style_table(analysis.high_correlations(corr, 0.8), precision=3))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>9 · Variáveis categóricas</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Primeiro perceber <b style='color:#3777FF'>o que contém cada variável</b> (categorias e peso de cada uma); depois a relação de cada categoria com o churn.</p></div></div>

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>9.1 · Dicionário de categorias</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Para cada variável: nº de categorias, lista de valores, moda e peso da moda.
Uma moda acima de ~90% (ex.: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Phone Service</code>) significa pouca variabilidade — a variável dificilmente discrimina.</p></div></div>

In [ ]:
cat_profile = analysis.categorical_profile(df_model, CAT_COLS)
cat_profile.to_csv(TAB_DIR / "04_perfil_categoricas.csv")
display(style_table(cat_profile, highlight="freq_moda"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>9.2 · Visão geral — composição de todas as categóricas</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Painel com uma mini-tabela visual por variável: cada barra é uma categoria, com a % e o nº de clientes.
Serve para ler rapidamente <b style='color:#0EE071'>que valores existem</b>, se há categorias raras (ex.: <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Mailed Check</code>) e se a variável está equilibrada.</p></div></div>

In [ ]:
plots.plot_categorical_overview(df_model, CAT_COLS, ncols=4, show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>9.3 · Detalhe por variável — composição e churn</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p>Para cada variável, dois painéis:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#3777FF'>Esquerda</b> — o que contém: nº e % de clientes por categoria</li><li><b style='color:#0EE071'>Direita</b> — barra 100%: proporção Stayed (<b style='color:#3777FF'>azul</b>) vs Churned (<b style='color:#0EE071'>verde</b>) em cada categoria; a linha tracejada marca o churn médio</li></ul>
<p>O <b style='color:#0EE071'>Cramér's V</b> (0 a 1) resume a força da associação com o churn: ≈0,1 fraca, ≈0,3 moderada, >0,5 forte.
Deriva do qui-quadrado, normalizado pelo tamanho da amostra e nº de categorias (versão com correção de Bergsma).
É simétrico e não implica causalidade — ex.: <i>Married</i> pode ser um proxy do tipo de contrato.</p></div></div>

In [ ]:
cramers = analysis.cramers_ranking(df_model, CAT_COLS, TARGET)
for col in CAT_COLS:
    plots.plot_categorical_detail(df_model, col, TARGET, overall_rate, cramers[col], show=SHOW)

In [ ]:
# Tabela completa: todas as categorias, nº, % e taxa de churn
value_dict = analysis.value_dictionary(df_model, CAT_COLS, TARGET)
value_dict.to_csv(TAB_DIR / "05_dicionario_categorias.csv", index=False)
display(style_table(value_dict, highlight="churn_rate"))

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>9.4 · Ranking Cramér's V</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'><p>Ordena as categóricas pela força da associação com o churn. Barras <b style='color:#0EE071'>verdes</b> = V ≥ 0,3.</p></div></div>

In [ ]:
cramers.to_csv(TAB_DIR / "06_cramers_v.csv")
plots.plot_ranking(cramers, "Associação categóricas × churn (Cramér's V)", "09_cramers_ranking",
                   highlight=set(cramers[cramers >= 0.3].index), show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>10 · Associação com o target e verificação de leakage</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>A <b style='color:#0EE071'>Mutual Information</b> mede quanta incerteza sobre o churn é reduzida ao conhecer cada variável;
ao contrário da correlação, capta relações não lineares e funciona para numéricas (<b style='color:#3777FF'>azul</b>) e categóricas (<b style='color:#0EE071'>verde</b>).</p></div></div>

In [ ]:
mi = analysis.mutual_information(df_model, NUM_COLS, CAT_COLS, TARGET)
mi.to_csv(TAB_DIR / "07_mutual_information.csv")
plots.plot_ranking(mi, "Mutual Information com o churn  (azul = numérica · verde = categórica)",
                   "10_mutual_information", highlight=set(CAT_COLS), show=SHOW)

<div style='background:#000000;border-left:6px solid #0EE071;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h3 style='color:#0EE071;margin:0 0 6px 0'>10.1 · Satisfaction Score e Churn Score</h3><div style='color:#FFFFFF;font-size:13.5px;line-height:1.6'>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> 1–2 corresponde a <b style='color:#0EE071'>100% de churn</b>: a variável parece ser registada no contexto da saída.
Incluída no modelo, inflaciona a performance sem ajudar a retenção preventiva. Sugestão: treinar com e sem, e reportar a diferença.</p>
<p><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Churn Score</code> é o modelo IBM existente — serve de <b style='color:#3777FF'>benchmark</b>: o AUC dele é a fasquia a bater.</p></div></div>

In [ ]:
plots.plot_leakage_checks(df_model, TARGET, show=SHOW)
sat = pd.crosstab(df_model["Satisfaction Score"], df_model["Customer Status"])
sat["churn_rate"] = sat["Churned"] / sat.sum(axis=1)
display(style_table(sat, highlight="churn_rate"))
print(f"AUC Churn Score IBM (população total): {roc_auc_score(df_model[TARGET], df_model['Churn Score']):.3f}")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>11 · Feature engineering</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Features calculadas <b style='color:#0EE071'>linha a linha</b> — não usam estatísticas do dataset, por isso podem ser criadas antes do split sem fuga de informação:</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>n_add_on_services</code> — nº de serviços adicionais subscritos</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>has_refund</code> — teve algum reembolso (Yes/No)</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>avg_hist_charge</code> — Total Charges ÷ Tenure (fatura média histórica)</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>charge_delta</code> — Monthly Charge − fatura média histórica (positivo ≈ aumento de preço recente)</li></ul></div></div>

In [ ]:
df_model = features.add_features(df_model)
df_joined = features.add_features(df_joined)
FEATURES_NUM = NUM_COLS + features.NEW_NUM
FEATURES_CAT = CAT_COLS + features.NEW_CAT

for col in features.NEW_NUM:
    plots.plot_numeric(df_model, col, TARGET, show=SHOW, prefix="11")
plots.plot_categorical_detail(df_model, "has_refund", TARGET, overall_rate,
                              analysis.cramers_v(df_model["has_refund"], df_model[TARGET]), show=SHOW)

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>12 · Split estratificado e pré-processadores</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Holdout <b style='color:#3777FF'>80/20 estratificado</b> ao nível do cliente (mantém a taxa de churn igual em treino e teste).
São definidos dois <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>ColumnTransformer</code>:</p>
<ul style='margin:6px 0 2px 0'><li><b style='color:#0EE071'>linear</b> — imputação + <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>StandardScaler</code> + one-hot (regressão logística, SVM, KNN)</li><li><b style='color:#0EE071'>trees</b> — imputação + one-hot, sem escala (Random Forest, Gradient Boosting, XGBoost, LightGBM)</li></ul>
<p>O <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>fit</code> é feito <b style='color:#0EE071'>apenas no treino</b>. O AUC do Churn Score IBM é recalculado no mesmo holdout para comparação justa.</p></div></div>

In [ ]:
X_train, X_test, y_train, y_test, idx_train, idx_test = preprocessing.stratified_split(
    df_model, FEATURES_NUM + FEATURES_CAT, TARGET)
print(f"Treino: {X_train.shape} | churn {y_train.mean():.1%}")
print(f"Teste : {X_test.shape} | churn {y_test.mean():.1%}")

preprocessors = preprocessing.build_preprocessors(FEATURES_NUM, FEATURES_CAT)
Xt = preprocessors["trees"].fit_transform(X_train)
print(f"Matriz final (árvores): {Xt.shape[1]} features")

auc_ibm_test = roc_auc_score(y_test, df_model.loc[idx_test, "Churn Score"])
print(f"AUC Churn Score IBM no holdout (benchmark): {auc_ibm_test:.3f}")

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>13 · Exportação</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<p>Ficheiros gerados (ignorados pelo Git — cada membro regenera-os ao correr o notebook):</p>
<ul style='margin:6px 0 2px 0'><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/train.csv</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>test.csv</code> — features + target + colunas de negócio (CLTV, Churn Score, Satisfaction Score)</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/joined_para_scoring.csv</code> — clientes Joined</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>data/processed/telco_merged_full.csv</code> — base unificada completa</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/artifacts/preprocessing_config.joblib</code> — pré-processadores e listas de features</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/figures/</code> e <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>outputs/tables/</code> — gráficos e tabelas da EDA</li></ul></div></div>

In [ ]:
business_cols = ["Customer ID", "CLTV", "Churn Score", "Satisfaction Score"]
cols_out = business_cols + FEATURES_NUM + FEATURES_CAT + [TARGET]
saved = preprocessing.export_artifacts(
    train=df_model.loc[idx_train, cols_out],
    test=df_model.loc[idx_test, cols_out],
    joined=df_joined,
    merged=df,
    config={"preprocessors": preprocessors, "features_num": FEATURES_NUM,
            "features_cat": FEATURES_CAT, "target": TARGET, "random_state": RANDOM_STATE},
)
for p in saved:
    print("✔", p.relative_to(ROOT))

<div style='background:#000000;border-left:6px solid #3777FF;border-radius:8px;padding:14px 20px;margin:4px 0;font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif'><h2 style='color:#3777FF;margin:0 0 8px 0'>14 · Próximos passos — ensemble</h2><div style='color:#FFFFFF;font-size:14px;line-height:1.6'>
<ul style='margin:6px 0 2px 0'><li>Carregar <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>preprocessing_config.joblib</code> e montar <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Pipeline([("prep", ...), ("clf", ...)])</code></li><li>Tuning com <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>StratifiedKFold</code> só no treino; o holdout usa-se uma única vez, no fim</li><li>Comparar com o AUC do <b style='color:#3777FF'>Churn Score IBM</b> no mesmo holdout</li><li>Testar <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>Satisfaction Score</code> com e sem, e documentar a diferença</li><li><code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>City</code>: se for testada, usar <code style='color:#0EE071;background:#111111;padding:1px 5px;border-radius:3px'>TargetEncoder</code> dentro do Pipeline</li><li>Priorização final: <b style='color:#0EE071'>P(churn) × CLTV</b> para ordenar a lista de intervenção</li></ul></div></div>